# UiA IKT Course Helper — Local RAG Chatbot

This notebook builds a small **retrieval-augmented generation (RAG)** chatbot for confused students who need help understanding **IKT courses at UiA**.

The notebook covers:

- scraping course pages from **UiA**
- parsing and cleaning the course descriptions
- chunking and indexing the material
- storing vectors in a local **ChromaDB**
- using **Hugging Face embeddings**
- using a local **Ollama** LLM
- running a simple CLI-style chatbot
- experimenting with retrieval depth (`k=1,3,7`), chunk sizes, overlap, and prompt styles


## 1. Environment

This notebook assumes:

- Python 3.10+
- `ollama` installed locally
- at least one Ollama chat model pulled, for example `llama3.1:8b`


Open a terminal and run this before using the chatbot:

```bash
ollama pull llama3.1:8b
```


In [3]:

# Install the main dependencies.
#%pip install -q     requests beautifulsoup4 lxml pandas tqdm chromadb     langchain langchain-community langchain-text-splitters     langchain-chroma langchain-huggingface langchain-ollama     sentence-transformers ipywidgets

## 2. Imports

In [5]:

from __future__ import annotations

import json
import os
import subprocess
import re
import textwrap
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Dict, Any, Iterable, Tuple

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
)
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

## 3. Configuration


In [7]:

CONFIG = {
    # Local folders
    "data_dir": Path("uia_ikt_rag_data"),
    "raw_jsonl": Path("uia_ikt_rag_data/uia_ikt_courses.jsonl"),
    "chroma_dir": Path("uia_ikt_rag_data/chroma_db"),

    # HTTP settings
    "timeout": 20,
    "sleep_between_requests": 0.5,
    "headers": {
        "User-Agent": "Mozilla/5.0 (compatible; UiA-IKT-RAG-Notebook/1.0)"
    },

    # A curated set of UiA IKT course pages.
    "seed_urls": [
        "https://www.uia.no/studier/emner/2025/host/ikt100.html",
        "https://www.uia.no/studier/emner/2025/host/ikt101.html",
        "https://www.uia.no/studier/emner/2025/host/ikt102.html",
        "https://www.uia.no/studier/emner/2026/var/ikt103.html",
        "https://www.uia.no/studier/emner/2026/var/ikt104.html",
        "https://www.uia.no/studier/emner/2026/var/ikt105.html",
        "https://www.uia.no/studier/emner/2026/var/ikt112.html",
        "https://www.uia.no/studier/emner/2026/var/ikt113.html",
        "https://www.uia.no/studier/emner/2026/var/ikt114.html",
        "https://www.uia.no/studier/emner/2026/var/ikt115.html",
        "https://www.uia.no/studier/emner/2025/host/ikt201.html",
        "https://www.uia.no/studier/emner/2025/host/ikt202.html",
        "https://www.uia.no/studier/emner/2025/host/ikt203.html",
        "https://www.uia.no/studier/emner/2026/var/ikt204.html",
        "https://www.uia.no/studier/emner/2026/var/ikt205.html",
        "https://www.uia.no/studier/emner/2026/var/ikt206.html",
        "https://www.uia.no/studier/emner/2025/host/ikt210.html",
        "https://www.uia.no/studier/emner/2025/host/ikt211.html",
        "https://www.uia.no/studier/emner/2025/host/ikt213.html",
        "https://www.uia.no/studier/emner/2025/host/ikt214.html",
        "https://www.uia.no/studier/emner/2025/host/ikt216.html",
        "https://www.uia.no/studier/emner/2025/host/ikt217.html",
        "https://www.uia.no/studier/emner/2026/var/ikt218.html",
        "https://www.uia.no/studier/emner/2026/var/ikt221.html",
        "https://www.uia.no/studier/emner/2025/host/ikt300.html",
        "https://www.uia.no/studier/emner/2026/var/ikt302.html",
        "https://www.uia.no/studier/emner/2025/host/ikt447.html",
        "https://www.uia.no/studier/emner/2025/host/ikt448.html",
        "https://www.uia.no/studier/emner/2026/var/ikt449.html",
        "https://www.uia.no/studier/emner/2025/host/ikt518.html",
        "https://www.uia.no/studier/emner/2026/var/ikt519.html",
        "https://www.uia.no/studier/emner/2025/host/ikt521.html",
        "https://www.uia.no/studier/emner/2025/host/ikt522.html",
        "https://www.uia.no/studier/emner/2026/var/ikt523.html",
        "https://www.uia.no/studier/emner/2025/host/ikt525.html",
        "https://www.uia.no/studier/emner/2025/host/ikt617.html",
        "https://www.uia.no/studier/emner/2025/host/ikt626.html",
        "https://www.uia.no/studier/emner/2025/host/ikt709.html",
        "https://www.uia.no/studier/emner/2025/host/ikt710.html",
        "https://www.uia.no/studier/emner/2025/host/ikt719.html",
        "https://www.uia.no/studier/emner/2025/host/ikt720.html",
        "https://www.uia.no/studier/emner/2026/var/ikt721.html",
        "https://www.uia.no/studier/emner/2026/var/ikt722.html",
        "https://www.uia.no/studier/emner/2026/var/ikt723.html",
        "https://www.uia.no/studier/emner/2026/var/ikt724.html",
        "https://www.uia.no/studier/emner/2026/var/ikt725.html",
        "https://www.uia.no/studier/emner/2026/var/ikt727.html",
        "https://www.uia.no/studier/emner/2025/host/ikt902.html",
    ],
    

    # Embeddings and LLM
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "ollama_model": "llama3.1:8b",

    # GPU settings
    "preferred_embedding_device": "cuda",
    "embedding_batch_size": 256,

    # Chunking defaults
    "chunk_method": "recursive",   # "recursive" or "markdown"
    "chunk_size": 900,
    "chunk_overlap": 120,

    # Retrieval defaults
    "default_k": 3,
}

CONFIG["data_dir"].mkdir(parents=True, exist_ok=True)
CONFIG["chroma_dir"].mkdir(parents=True, exist_ok=True)

CONFIG

# Expected courses retrieved by model, for use in table generation
EXPECTED_COURSES = {
    1: {
        "primary": {"IKT115", "IKT902", "IKT112"},
        "secondary": {"IKT724", "IKT725"},
        "unsupported": False,
    },
    2: {
        "primary": {"IKT300"},
        "secondary": set(),
        "unsupported": False,
    },
    3: {
        "primary": {"IKT112", "IKT115", "IKT902", "IKT724", "IKT725"},
        "secondary": {"IKT710"},
        "unsupported": False,
    },
    4: {
        "primary": set(),
        "secondary": set(),
        "unsupported": True,
    },
    5: {
        "primary": {"IKT448"},
        "secondary": {"IKT626", "IKT447", "IKT449", "IKT518", "IKT521", "IKT522"},
        "unsupported": False,
    },
}

## GPU diagnostics and acceleration


In [9]:
def detect_embedding_device(preferred: str = "cuda") -> str:
    """Return 'cuda' if requested and available, otherwise 'cpu'."""
    if preferred != "cuda":
        return "cpu"

    try:
        import torch
        # Print CUDA information to make it clear whether embeddings can run on GPU.
        print(f"PyTorch version: {torch.__version__}")
        print(f"PyTorch CUDA build: {torch.version.cuda}")
        print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")

        if torch.cuda.is_available():
            device_name = torch.cuda.get_device_name(0)
            capability = torch.cuda.get_device_capability(0)
            print(f"CUDA device 0: {device_name}")
            print(f"CUDA compute capability: {capability[0]}.{capability[1]}")
            return "cuda"

    except Exception as e:
        print(f"Could not inspect PyTorch/CUDA: {e}")
    
    # CPU is used as a safe fallback if CUDA is unavailable or cannot be inspected.
    print("Falling back to CPU for embeddings.")
    return "cpu"


CONFIG["embedding_device"] = detect_embedding_device(CONFIG.get("preferred_embedding_device", "cuda"))

def make_embedding_model(model_name: str | None = None) -> HuggingFaceEmbeddings:
    """Create the embedding model on GPU when CUDA is available."""
    model_name = model_name or CONFIG["embedding_model"]
    device = CONFIG.get("embedding_device", "cpu")
    batch_size = int(CONFIG.get("embedding_batch_size", 64))

    print(
        f"Loading embedding model '{model_name}' on device='{device}' "
        f"with batch_size={batch_size}",
        flush=True,
    )

    return HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={"device": device},
        encode_kwargs={
            # Normalized embeddings make similarity search more stable for this setup.
            "normalize_embeddings": True,
            "batch_size": batch_size,
        },
    )


def print_ollama_status() -> None:
    """Show whether Ollama has loaded the model on GPU or CPU.

    Run this after at least one llm.invoke(...) call.
    In the PROCESSOR column, you want to see something like '100% GPU'.
    """
    try:
        # This is only a runtime check; it does not affect the experiment results.
        result = subprocess.run(
            ["ollama", "ps"],
            capture_output=True,
            text=True,
            timeout=10,
        )
        if result.returncode == 0:
            print(result.stdout)
        else:
            print(result.stderr)
    except Exception as e:
        print(f"Could not run 'ollama ps': {e}")


PyTorch version: 2.10.0+cu130
PyTorch CUDA build: 13.0
torch.cuda.is_available(): True
CUDA device 0: NVIDIA GeForce RTX 5090
CUDA compute capability: 12.0


## 4. Scraper helpers

This scraper uses **Requests + Beautiful Soup** and tries to extract:

- course code and title
- URL
- visible text sections
- metadata-like lines such as credits, language, semester, prerequisites, and learning outcomes


In [11]:

@dataclass
class CoursePage:
    # Small container for the scraped information from one UiA course page.
    url: str
    title: str
    course_code: str
    course_name: str
    text: str
    sections: Dict[str, str]


def fetch_html(url: str, timeout: int = 20, headers: dict | None = None) -> str:
    # Fetch the raw HTML and fail early if the request was not successful.
    response = requests.get(url, timeout=timeout, headers=headers or {})
    response.raise_for_status()
    return response.text


def clean_text(text: str) -> str:
    # Normalize whitespace so scraped text becomes easier to chunk and embed.
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\u00a0", " ", text)
    return text.strip()


def extract_course_code_and_name(page_title: str) -> tuple[str, str]:
    # Typical title pattern on UiA pages:
    # "IKT115 Introduksjon til kunstig intelligens-teknologi (Vår 2026)"
    m = re.match(r"^([A-ZÆØÅ\-]{2,}\d{3}[A-Z]?)\s+(.*?)(?:\s*\(|$)", page_title.strip())
    if m:
        return m.group(1).strip(), m.group(2).strip()
    return "", page_title.strip()


def parse_uia_course_page(html: str, url: str) -> CoursePage:
    soup = BeautifulSoup(html, "lxml")

    # Remove page elements that do not contain useful course information.
    for tag in soup(["script", "style", "noscript", "svg", "footer"]):
        tag.decompose()

    # Try to find a strong page title first
    title_candidates = []
    if soup.title and soup.title.get_text(strip=True):
        title_candidates.append(soup.title.get_text(" ", strip=True))

    for selector in ["h1", "main h1", ".page-title", ".article h1"]:
        node = soup.select_one(selector)
        if node:
            title_candidates.append(node.get_text(" ", strip=True))

    page_title = next((t for t in title_candidates if t), "UiA course page")
    course_code, course_name = extract_course_code_and_name(page_title)

    # Extract headings and the following text blocks
    headings = soup.find_all(["h1", "h2", "h3"])
    sections: Dict[str, str] = {}

    for heading in headings:
        heading_text = clean_text(heading.get_text(" ", strip=True))
        if not heading_text:
            continue

        collected = []
        for sib in heading.find_next_siblings():
            if sib.name in ["h1", "h2", "h3"]:
                break
            text = clean_text(sib.get_text(" ", strip=True))
            if text:
                collected.append(text)

        if collected:
            sections[heading_text] = " ".join(collected)

    # Fallback: if section extraction is poor, use visible text from main/article/body
    main_node = soup.select_one("main") or soup.select_one("article") or soup.body
    visible_text = clean_text(main_node.get_text("\n", strip=True)) if main_node else ""

    # Build a compact but useful text representation
    joined_sections = []
    for k, v in sections.items():
        joined_sections.append(f"{k}\n{v}")

    if joined_sections:
        combined_text = f"{page_title}\n\n" + "\n\n".join(joined_sections)
    else:
        combined_text = f"{page_title}\n\n{visible_text}"

    return CoursePage(
        url=url,
        title=page_title,
        course_code=course_code,
        course_name=course_name,
        text=combined_text,
        sections=sections,
    )

## 5. Scrape the UiA IKT course pages

Collect the seed pages and store them as JSON Lines (`.jsonl`) so the raw scrape is easy to inspect or reuse.

In [13]:

def scrape_seed_pages(config: dict) -> list[CoursePage]:
    pages: list[CoursePage] = []

    # Scrape the fixed list of UiA course URLs used as the corpus.
    for url in tqdm(config["seed_urls"], desc="Scraping UiA course pages"):
        try:
            html = fetch_html(
                url,
                timeout=config["timeout"],
                headers=config["headers"],
            )
            page = parse_uia_course_page(html, url)
            pages.append(page)
            # Add a short pause to avoid sending requests too quickly.
            time.sleep(config["sleep_between_requests"])
        except Exception as e:
            print(f"Failed to scrape {url}: {e}")

    return pages

def load_pages_from_jsonl(path: Path) -> list[CoursePage]:
    # Load the cached corpus so repeated experiments use the same data snapshot.
    pages = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            pages.append(CoursePage(**json.loads(line)))
    return pages


FORCE_RESCRAPE = False

if CONFIG["raw_jsonl"].exists() and not FORCE_RESCRAPE:
    # Reuse the cached scrape unless a fresh scrape is explicitly requested.
    pages = load_pages_from_jsonl(CONFIG["raw_jsonl"])
    print(f"Loaded {len(pages)} pages from cache: {CONFIG['raw_jsonl']}")
else:
    pages = scrape_seed_pages(CONFIG)
    # Store the scraped pages as JSONL so later runs are reproducible.
    with open(CONFIG["raw_jsonl"], "w", encoding="utf-8") as f:
        for page in pages:
            f.write(json.dumps(asdict(page), ensure_ascii=False) + "\n")

    print(f"Scraped and saved {len(pages)} pages to: {CONFIG['raw_jsonl']}")
    
# Save the scraped pages to JSONL.
with open(CONFIG["raw_jsonl"], "w", encoding="utf-8") as f:
    for page in pages:
        f.write(json.dumps(asdict(page), ensure_ascii=False) + "\n")


Loaded 48 pages from cache: uia_ikt_rag_data\uia_ikt_courses.jsonl


In [14]:

# Quick inspection
df_pages = pd.DataFrame([asdict(p) for p in pages])
df_pages[["course_code", "course_name", "url"]]

,course_code,course_name,url
0,IKT100,"Nettverk, sikkerhet og personvern",https://www.uia.no/studier/emner/2025/host/ikt...
1,IKT101,Grunnleggende softwareutvikling,https://www.uia.no/studier/emner/2025/host/ikt...
2,IKT102,Operativsystemer,https://www.uia.no/studier/emner/2025/host/ikt...
3,IKT103,Videregående softwareutvikling,https://www.uia.no/studier/emner/2026/var/ikt1...
4,IKT104,Mikrokontrollere,https://www.uia.no/studier/emner/2026/var/ikt1...
5,IKT105,Datamodellering og databaser,https://www.uia.no/studier/emner/2026/var/ikt1...
6,IKT112,Concepts of Machine Learning,https://www.uia.no/studier/emner/2026/var/ikt1...
7,IKT113,Cybersikkerhet,https://www.uia.no/studier/emner/2026/var/ikt1...
8,IKT114,IT-orkestrering,https://www.uia.no/studier/emner/2026/var/ikt1...
9,IKT115,Introduksjon til kunstig intelligens-teknologi,https://www.uia.no/studier/emner/2026/var/ikt1...


## 6. Convert the scraped pages into LangChain documents

Each course page becomes one base document with metadata.

That metadata is useful for:
- displaying where the answer came from
- filtering by course code
- debugging retrieval quality

In [16]:

def load_documents_from_pages(pages: list[CoursePage]) -> list[Document]:
    # Convert scraped course pages into LangChain documents for the RAG pipeline.
    docs = []

    for page in pages:
        # Metadata is kept with each document so retrieved chunks can be traced
        # back to the original course page during evaluation and answer display.
        metadata = {
            "source": page.url,
            "course_code": page.course_code,
            "course_name": page.course_name,
            "title": page.title,
        }
        docs.append(Document(page_content=page.text, metadata=metadata))

    return docs


base_docs = load_documents_from_pages(pages)
len(base_docs)

48

In [17]:

# Show one example document
print(base_docs[0].metadata)
print()
print(base_docs[0].page_content[:1500])

{'source': 'https://www.uia.no/studier/emner/2025/host/ikt100.html', 'course_code': 'IKT100', 'course_name': 'Nettverk, sikkerhet og personvern', 'title': 'IKT100 Nettverk, sikkerhet og personvern (Høst 2025) - Universitetet i Agder'}

IKT100 Nettverk, sikkerhet og personvern (Høst 2025) - Universitetet i Agder

Emnet er tilknyttet følgende studieprogram
Ingeniørfag - data, bachelorprogram Ingeniørfag - elektronikk, bachelorprogram

Undervisningsspråk
Emnet undervises på norsk.

Læringsutbytte
Etter fullført emne skal studenten ha kjennskap til de viktigste nettverksprotokollene ha kjennskap til sikkerhetsbegreper som autentisering og autorisering, integritet, konfidensialitet og tilgjengelighet kjenne til gjeldende personvernregelverk forstå hvordan nettverk er bygget opp med switcher, rutere og aksesspunkter kunne vurdere enkle trusselbilder kunne gjennomføre enkle trussel- og risikoanalyser

Innhold
Innføring i protokoller og nettverkskomponenter som svitsjer, rutere, kabler og mask

## 7. Chunking functions

Chunking styles:

1. **Recursive character chunking**
   - simple and robust
   - often a strong baseline

 
2. **Markdown-like heading chunking**
   - tries to preserve section structure
   - can be useful when course pages have meaningful headings


In [19]:
def chunk_docs_recursive(
    docs: list[Document],
    chunk_size: int = 900,
    chunk_overlap: int = 120,
) -> list[Document]:
    # Recursive chunking is the baseline method used for general plain text.
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    return splitter.split_documents(docs)


def chunk_docs_markdown_like(
    docs: list[Document],
    chunk_size: int = 900,
    chunk_overlap: int = 120,
) -> list[Document]:
    # This method tries to preserve section-like structure before recursive splitting.
    header_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ("#", "Header 1"),
            ("##", "Header 2"),
            ("###", "Header 3"),
        ]
    )

    recursively_chunked = []

    for doc in docs:
        # Convert detected headings to rough markdown before splitting.
        pseudo_markdown = doc.page_content
        pseudo_markdown = re.sub(r"(?m)^([A-ZÆØÅa-zæøå0-9 ,\-_/]+)\n", r"## \1\n", pseudo_markdown)

        md_docs = header_splitter.split_text(pseudo_markdown)
        md_docs = [
            Document(
                page_content=d.page_content,
                metadata={**doc.metadata, **d.metadata}
            )
            for d in md_docs
        ]

        # Apply recursive splitting after section splitting to keep chunks within size limits.
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ". ", " ", ""],
        )
        recursively_chunked.extend(splitter.split_documents(md_docs))

    return recursively_chunked


def chunk_documents(
    docs: list[Document],
    method: str = "recursive",
    chunk_size: int = 900,
    chunk_overlap: int = 120,
) -> list[Document]:
    # Select chunking strategy so it can be varied in the experiment grid.
    if method == "recursive":
        return chunk_docs_recursive(docs, chunk_size, chunk_overlap)
    elif method == "markdown":
        return chunk_docs_markdown_like(docs, chunk_size, chunk_overlap)
    else:
        raise ValueError(f"Unknown chunking method: {method}")

In [20]:
# Create chunks using the default settings from CONFIG.
chunked_docs = chunk_documents(
    base_docs,
    method=CONFIG["chunk_method"],
    chunk_size=CONFIG["chunk_size"],
    chunk_overlap=CONFIG["chunk_overlap"],
)

# Print counts as a quick sanity check before building the vector store.
print(f"Base docs: {len(base_docs)}")
print(f"Chunked docs: {len(chunked_docs)}")

Base docs: 48
Chunked docs: 200


In [21]:

# Inspect a few chunks
for i, doc in enumerate(chunked_docs[:3], start=1):
    print("=" * 90)
    print(f"Chunk {i}")
    print(doc.metadata)
    print(doc.page_content[:1000])
    print()

Chunk 1
{'source': 'https://www.uia.no/studier/emner/2025/host/ikt100.html', 'course_code': 'IKT100', 'course_name': 'Nettverk, sikkerhet og personvern', 'title': 'IKT100 Nettverk, sikkerhet og personvern (Høst 2025) - Universitetet i Agder'}
IKT100 Nettverk, sikkerhet og personvern (Høst 2025) - Universitetet i Agder

Emnet er tilknyttet følgende studieprogram
Ingeniørfag - data, bachelorprogram Ingeniørfag - elektronikk, bachelorprogram

Undervisningsspråk
Emnet undervises på norsk.

Læringsutbytte
Etter fullført emne skal studenten ha kjennskap til de viktigste nettverksprotokollene ha kjennskap til sikkerhetsbegreper som autentisering og autorisering, integritet, konfidensialitet og tilgjengelighet kjenne til gjeldende personvernregelverk forstå hvordan nettverk er bygget opp med switcher, rutere og aksesspunkter kunne vurdere enkle trusselbilder kunne gjennomføre enkle trussel- og risikoanalyser

Chunk 2
{'source': 'https://www.uia.no/studier/emner/2025/host/ikt100.html', 'course_

## 8. Create local embeddings and a Chroma vector store

This notebook uses:

- **Hugging Face embeddings** (`all-MiniLM-L6-v2`) for a fast local baseline
- **Chroma** as a local persistent vector database


In [23]:
# Create the embedding model used to convert text chunks into vectors.
embedding_model = make_embedding_model(CONFIG["embedding_model"])

# Open the Chroma collection so it can be reset before adding new chunks.
vectorstore = Chroma(
    collection_name="uia_ikt_courses",
    persist_directory=str(CONFIG["chroma_dir"]),
    embedding_function=embedding_model,
)

try:
    # Delete any old collection so this run only contains the current chunks.
    vectorstore.delete_collection()
except Exception:
    pass

# Recreate the collection after deletion.
vectorstore = Chroma(
    collection_name="uia_ikt_courses",
    persist_directory=str(CONFIG["chroma_dir"]),
    embedding_function=embedding_model,
)

# Add the chunked course documents to the vector database.
vectorstore.add_documents(chunked_docs)
print("Vector store is ready.")

Loading embedding model 'sentence-transformers/all-MiniLM-L6-v2' on device='cuda' with batch_size=256
Vector store is ready.


## 9. Retrieval helper

This function shows what the retriever actually returns.

In [25]:
def retrieve_context(
    query: str,
    vectorstore: Chroma,
    k: int = 3,
) -> list[Document]:
    # Retrieve the k most similar chunks for a user query.
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    return retriever.invoke(query)


# Quick retrieval test before connecting the retriever to the LLM.
test_query = "What does IKT114 cover?"
retrieved = retrieve_context(test_query, vectorstore, k=3)

# Inspect retrieved metadata and text to check whether the results look relevant.
for i, doc in enumerate(retrieved, start=1):
    print("=" * 100)
    print(f"Result {i}")
    print(doc.metadata)
    print(doc.page_content[:1000])
    print()

Result 1
{'course_code': 'IKT214', 'title': 'IKT214 Sikkerhet og IoT (Høst 2025) - Universitetet i Agder', 'source': 'https://www.uia.no/studier/emner/2025/host/ikt214.html', 'course_name': 'Sikkerhet og IoT'}
IKT214 Sikkerhet og IoT (Høst 2025) - Universitetet i Agder

Undervisningsspråk
Norsk eller engelsk. Emnet undervises på engelsk dersom det er utvekslingsstudenter som følger emnet.

Anbefalte forkunnskaper
Emnet forutsetter forkunnskaper tilsvarende IKT100-G Nettverk, Sikkerhet og Personvern, IKT101-G Grunnleggende softwareutvikling, IKT204-G Datakommunikasjon.

Result 2
{'source': 'https://www.uia.no/studier/emner/2026/var/ikt727.html', 'course_code': 'IKT727', 'title': 'IKT727 Advanced Theory on Selected Topics in ICT (Vår 2026) - Universitetet i Agder', 'course_name': 'Advanced Theory on Selected Topics in ICT'}
IKT727 Advanced Theory on Selected Topics in ICT (Vår 2026) - Universitetet i Agder

Emnet er tilknyttet følgende studieprogram
Ph.d.-program i teknologi og realfag



## 10. Build the vanilla RAG chain

1. retrieve top-`k` chunks
2. build a prompt from the retrieved context
3. ask a local Ollama model to answer

The system prompt tells the model to:
- stay grounded in the retrieved course data
- admit uncertainty
- mention the course code when relevant
- avoid inventing facts

In [27]:
# Default prompt used for grounded answers based on retrieved course context.
DEFAULT_SYSTEM_PROMPT = """You are a helpful academic chatbot for students at the University of Agder (UiA).
You answer questions about IKT-related courses using ONLY the supplied context.

Rules:
- Be precise and practical.
- If the context is incomplete, say so clearly.
- Prefer course codes and course names when relevant.
- Do not invent prerequisites, deadlines, exams, or course details.
- If multiple courses may match, compare them briefly.
- End with a short 'Sources:' line listing course codes or URLs from the retrieved context.
"""

# Prompt template that combines the system instructions, user question, and retrieved context.
RAG_PROMPT = ChatPromptTemplate.from_messages(
    [
        ("system", "{system_prompt}"),
        ("human", """Question:
{question}

Retrieved context:
{context}

Answer the question in clear English for a confused student.
"""),
    ]
)

# Temperature is set to 0 to make the experiment more reproducible.
llm = ChatOllama(
    model=CONFIG["ollama_model"],
    temperature=0,
    num_ctx=4096,
    keep_alive="10m",
)

In [28]:
def format_context(docs: list[Document]) -> str:
    # Format retrieved chunks with metadata so the LLM can see where each chunk comes from.
    blocks = []
    for i, doc in enumerate(docs, start=1):
        meta = doc.metadata
        header = (
            f"[Document {i}] "
            f"course_code={meta.get('course_code', '')}; "
            f"course_name={meta.get('course_name', '')}; "
            f"source={meta.get('source', '')}"
        )
        blocks.append(header + "\n" + doc.page_content)

    return "\n\n" + ("\n\n" + "-" * 80 + "\n\n").join(blocks)


def answer_question(
    question: str,
    vectorstore: Chroma,
    llm: ChatOllama,
    k: int = 3,
    system_prompt: str = DEFAULT_SYSTEM_PROMPT,
) -> tuple[str, list[Document]]:
    # Retrieve context first, then pass both the question and context to the LLM.
    docs = retrieve_context(question, vectorstore, k=k)
    context = format_context(docs)

    messages = RAG_PROMPT.invoke(
        {
            "system_prompt": system_prompt,
            "question": question,
            "context": context,
        }
    )

    response = llm.invoke(messages)

    # Return both the generated answer and retrieved documents for later evaluation.
    return response.content, docs

## 11. Test the chatbot on one question

In [30]:
# Simple manual test of the full RAG pipeline with one student-style question.
question = "I am new to AI. Which UiA IKT course looks most relevant as a starting point?"

answer, used_docs = answer_question(
    question=question,
    vectorstore=vectorstore,
    llm=llm,
    k=3,
)

print(answer)

Based on your background as a beginner in AI, I would recommend starting with **IKT112: Concepts of Machine Learning**. This course covers the fundamental concepts of machine learning, including supervised and unsupervised learning, reinforcement learning, and real-world applications.

The course is designed to provide a basic understanding of machine learning, which will give you a solid foundation for further studies in AI. The course also includes hands-on experience with applying machine learning algorithms using software packages like WEKA.

In contrast, **IKT724: Deep Learning for Engineers** seems more advanced and focused on deep learning techniques, which may be too specialized for a beginner.

Sources:
https://www.uia.no/studier/emner/2026/var/ikt112.html
https://www.uia.no/studier/emner/2026/var/ikt724.html


## 12. Chat test

In [32]:
def answer_question(
    question: str,
    vectorstore: Chroma,
    llm: ChatOllama,
    k: int = 3,
    system_prompt: str = DEFAULT_SYSTEM_PROMPT,
):
    # Use direct similarity search here to retrieve the top k chunks from Chroma.
    docs = vectorstore.similarity_search(question, k=k)

    context_parts = []
    for doc in docs:
        # Include course metadata in the context so the model can cite course codes.
        source = doc.metadata.get("course_code", "Unknown course")
        url = doc.metadata.get("url", "No URL")
        text = doc.page_content[:800]
        context_parts.append(f"Course: {source}\nURL: {url}\n{text}")

    context = "\n\n".join(context_parts)

    # The prompt tells the model to stay within the retrieved context.
    prompt = f"""
{system_prompt}

Use the context below to answer the question.
If the answer is not clearly supported by the context, say so.

Context:
{context}

Question: {question}
Answer:
""".strip()

    response = llm.invoke(prompt)

    if hasattr(response, "content"):
        answer = response.content
    else:
        answer = str(response)

    # Return retrieved documents as well, since they are used later for evaluation.
    return answer, docs


# Manual test question to check that retrieval and generation work together.
answer, docs = answer_question(
    question="Tell me about IKT300.",
    vectorstore=vectorstore,
    llm=llm,
    k=3,
)

print("Bot:", answer)

Bot: Based on the provided context, here is what I can tell you about IKT300:

IKT300 is a course called "Software arkitektur og design" (Software Architecture and Design) offered at UiA in the fall of 2025. It is part of the 5-årig masterprogram in Kunstig intelligens (Artificial Intelligence). The course language can be either Norwegian or English, but will be taught in English if there are exchange students following the course.

The recommended prerequisites for this course include IKT101-G Grunnleggende softwareutvikling (Basic Software Development), IKT103 Videregående softwareutvikling (Advanced Software Development), DAT220 or equivalent, and IKT202-G Software Engineering.

Sources:
IKT300


### Verify whether Ollama uses the GPU


In [34]:
print_ollama_status()


NAME           ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
llama3.1:8b    46e0c10c039e    5.5 GB    100% GPU     4096       9 minutes from now    



## 13. Parameter experiments

- `k=1,3,7`
- chunking methods
- system prompt style
- other relevant parameters


In [36]:
# Fixed question set used for all experiment configurations.
EXPERIMENT_QUESTIONS = [
    "What course should I take if I want to learn AI?",
    "Which course is about software architecture?",
    "What are the differences between the IKT courses related to AI and machine learning?",
    "Which IKT course gives the best job opportunities?",
    "Which course is most related to cybersecurity research methods?",
]

# Prompt variants used to test how prompt style affects answer generation.
# Retrieval is unchanged; only the LLM instructions differ.
PROMPT_VARIANTS = {
    "strict_grounded": DEFAULT_SYSTEM_PROMPT,
    "friendly_advisor": """You are a friendly UiA study assistant.
Use only the retrieved context.
Explain things simply for a confused first-year student.
Be honest when the data is incomplete.
End with a short 'Sources:' line.
""",
    "compare_courses": """You are a helpful course advisor for UiA IKT students.
Use only the retrieved context.
When relevant, compare courses side by side and explain differences.
Do not guess missing details.
End with a short 'Sources:' line.
""",
}

print("Done!")

Done!


In [37]:
def rebuild_vectorstore_for_experiment(
    docs: list[Document],
    embedding_model_name: str,
    persist_dir: Path,
    collection_name: str,
    chunk_method: str,
    chunk_size: int,
    chunk_overlap: int,
) -> tuple[Chroma, list[Document]]:
    # Each experiment configuration gets its own embedding model and vector store.
    embedding_model = make_embedding_model(embedding_model_name)

    # Rebuild chunks so chunking method, size, and overlap can be tested separately.
    chunked = chunk_documents(
        docs,
        method=chunk_method,
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    vectorstore = Chroma(
        collection_name=collection_name,
        persist_directory=str(persist_dir),
        embedding_function=embedding_model,
    )

    try:
        # Clear any old data from the same collection before adding new chunks.
        vectorstore.delete_collection()
    except Exception:
        pass

    # Recreate the collection after deletion.
    vectorstore = Chroma(
        collection_name=collection_name,
        persist_directory=str(persist_dir),
        embedding_function=embedding_model,
    )

    # Store the current configuration's chunks in Chroma.
    vectorstore.add_documents(chunked)
    return vectorstore, chunked


print("Done!")

Done!


In [38]:
def run_experiment_grid(
    docs: list[Document],
    embedding_model_name: str,
    chroma_root: Path,
    llm: ChatOllama,
    questions: list[str],
    k_values: list[int],
    chunk_methods: list[str],
    chunk_sizes: list[int],
    chunk_overlaps: list[int],
    prompt_variants: dict[str, str],
    partial_results_path: Path | None = None,
    save_every: int = 1,
    resume: bool = True,
) -> pd.DataFrame:
    """Run the full RAG experiment grid with progress logging and incremental saving."""

    def format_duration(seconds: float | int | None) -> str:
        if seconds is None:
            return "?"
        seconds = max(0, int(seconds))
        hours, remainder = divmod(seconds, 3600)
        minutes, seconds = divmod(remainder, 60)
        if hours:
            return f"{hours}h {minutes}m {seconds}s"
        if minutes:
            return f"{minutes}m {seconds}s"
        return f"{seconds}s"

    def make_key(
        chunk_method: str,
        chunk_size: int,
        chunk_overlap: int,
        prompt_name: str,
        k: int,
        question: str,
    ) -> tuple:
        # Unique key used to skip calls that already exist in the partial CSV.
        return (
            chunk_method,
            int(chunk_size),
            int(chunk_overlap),
            prompt_name,
            int(k),
            question,
        )

    def get_retrieval_metrics(retrieved_docs: list[Document], question_index: int) -> dict:
        """Compute source diversity and expected-course retrieval metrics."""
        retrieved_sources = [
            doc.metadata.get("course_code", "")
            for doc in retrieved_docs
        ]

        # Unique course codes are used to measure source diversity.
        unique_sources = set(src for src in retrieved_sources if src)

        expected = EXPECTED_COURSES.get(question_index, {
            "primary": set(),
            "secondary": set(),
            "unsupported": False,
        })

        primary_expected = expected["primary"]
        secondary_expected = expected["secondary"]
        all_expected = primary_expected | secondary_expected

        primary_found = unique_sources & primary_expected
        secondary_found = unique_sources & secondary_expected
        all_found = unique_sources & all_expected

        # Duplicate ratio shows how much retrieved context comes from repeated courses.
        duplicate_ratio = (
            round(1 - (len(unique_sources) / len(retrieved_sources)), 4)
            if retrieved_sources else None
        )

        primary_recall = (
            round(len(primary_found) / len(primary_expected), 4)
            if primary_expected else None
        )

        expected_recall = (
            round(len(all_found) / len(all_expected), 4)
            if all_expected else None
        )

        # For the comparison question, at least three primary courses gives useful coverage.
        comparison_coverage_good = (
            len(primary_found) >= 3
            if question_index == 3
            else None
        )

        return {
            "retrieved_sources": ";".join(retrieved_sources),
            "num_sources": len(unique_sources),
            "duplicate_ratio": duplicate_ratio,
            "expected_primary_courses": ";".join(sorted(primary_expected)),
            "expected_secondary_courses": ";".join(sorted(secondary_expected)),
            "contains_primary_expected": bool(primary_found),
            "contains_any_expected": bool(all_found),
            "primary_courses_found": ";".join(sorted(primary_found)),
            "secondary_courses_found": ";".join(sorted(secondary_found)),
            "num_primary_courses_found": len(primary_found),
            "num_expected_courses_found": len(all_found),
            "primary_recall": primary_recall,
            "expected_recall": expected_recall,
            "is_unsupported_question": expected["unsupported"],
            "comparison_coverage_good": comparison_coverage_good,
        }

    def empty_retrieval_metrics(question_index: int) -> dict:
        """Return empty retrieval metrics for failed runs."""
        expected = EXPECTED_COURSES.get(question_index, {
            "primary": set(),
            "secondary": set(),
            "unsupported": False,
        })

        return {
            "retrieved_sources": "",
            "num_sources": 0,
            "duplicate_ratio": None,
            "expected_primary_courses": ";".join(sorted(expected["primary"])),
            "expected_secondary_courses": ";".join(sorted(expected["secondary"])),
            "contains_primary_expected": False,
            "contains_any_expected": False,
            "primary_courses_found": "",
            "secondary_courses_found": "",
            "num_primary_courses_found": 0,
            "num_expected_courses_found": 0,
            "primary_recall": None,
            "expected_recall": None,
            "is_unsupported_question": expected["unsupported"],
            "comparison_coverage_good": None,
        }

    key_columns = [
        "chunk_method",
        "chunk_size",
        "chunk_overlap",
        "prompt_name",
        "k",
        "question",
    ]

    total_vectorstores = len(chunk_methods) * len(chunk_sizes) * len(chunk_overlaps)
    total_calls = (
        total_vectorstores
        * len(prompt_variants)
        * len(k_values)
        * len(questions)
    )

    partial_results_path = Path(partial_results_path) if partial_results_path is not None else None
    if partial_results_path is not None:
        partial_results_path.parent.mkdir(parents=True, exist_ok=True)

    rows: list[dict] = []
    completed_keys: set[tuple] = set()

    # Resume mode avoids repeating expensive LLM calls after an interrupted run.
    if resume and partial_results_path is not None and partial_results_path.exists():
        previous_df = pd.read_csv(partial_results_path)
        rows = previous_df.to_dict("records")

        missing_columns = [col for col in key_columns if col not in previous_df.columns]
        if missing_columns:
            print(
                f"Existing partial file is missing key columns {missing_columns}; "
                "resume is disabled for this run.",
                flush=True,
            )
        else:
            completed_keys = {
                make_key(
                    row["chunk_method"],
                    row["chunk_size"],
                    row["chunk_overlap"],
                    row["prompt_name"],
                    row["k"],
                    row["question"],
                )
                for _, row in previous_df.iterrows()
            }
            print(
                f"Loaded {len(previous_df)} existing rows from {partial_results_path}. "
                f"Skipping {len(completed_keys)} completed experiment calls.",
                flush=True,
            )

    start_time = time.time()
    initial_completed = len(completed_keys)

    print("=" * 80, flush=True)
    print("Starting RAG experiment grid", flush=True)
    print(f"Embedding device:           {CONFIG.get('embedding_device', 'unknown')}", flush=True)
    print(f"Vectorstore builds planned: {total_vectorstores}", flush=True)
    print(f"LLM calls planned:          {total_calls}", flush=True)
    print(f"Already completed:          {initial_completed}", flush=True)
    if partial_results_path is not None:
        print(f"Partial CSV:                {partial_results_path}", flush=True)
        print(f"Save frequency:             every {save_every} completed call(s)", flush=True)
    print("=" * 80, flush=True)

    completed_this_run = 0
    vectorstore_index = 0

    try:
        for chunk_method in chunk_methods:
            for chunk_size in chunk_sizes:
                for chunk_overlap in chunk_overlaps:
                    vectorstore_index += 1
                    collection_name = f"uia_ikt_{chunk_method}_{chunk_size}_{chunk_overlap}".replace("-", "_")

                    pending_calls = []
                    for prompt_name, prompt_text in prompt_variants.items():
                        for k in k_values:
                            for question_index, question in enumerate(questions, start=1):
                                key = make_key(
                                    chunk_method,
                                    chunk_size,
                                    chunk_overlap,
                                    prompt_name,
                                    k,
                                    question,
                                )

                                if key in completed_keys:
                                    continue

                                pending_calls.append(
                                    {
                                        "key": key,
                                        "prompt_name": prompt_name,
                                        "prompt_text": prompt_text,
                                        "k": k,
                                        "question": question,
                                        "question_index": question_index,
                                    }
                                )

                    if not pending_calls:
                        print(
                            f"[Vectorstore {vectorstore_index}/{total_vectorstores}] "
                            f"Skipping {collection_name}; all calls already completed.",
                            flush=True,
                        )
                        continue

                    print("-" * 80, flush=True)
                    print(
                        f"[Vectorstore {vectorstore_index}/{total_vectorstores}] "
                        f"Building {collection_name} "
                        f"(method={chunk_method}, size={chunk_size}, overlap={chunk_overlap})",
                        flush=True,
                    )

                    vectorstore_started = time.time()

                    try:
                        # Rebuild the vector store once per chunking configuration.
                        vectorstore, chunked = rebuild_vectorstore_for_experiment(
                            docs=docs,
                            embedding_model_name=embedding_model_name,
                            persist_dir=chroma_root / collection_name,
                            collection_name=collection_name,
                            chunk_method=chunk_method,
                            chunk_size=chunk_size,
                            chunk_overlap=chunk_overlap,
                        )

                        vectorstore_seconds = time.time() - vectorstore_started

                        print(
                            f"[Vectorstore {vectorstore_index}/{total_vectorstores}] "
                            f"Ready with {len(chunked)} chunks in {format_duration(vectorstore_seconds)}. "
                            f"Pending LLM calls for this vectorstore: {len(pending_calls)}",
                            flush=True,
                        )

                    except Exception as e:
                        vectorstore_seconds = time.time() - vectorstore_started

                        print(
                            f"[Vectorstore {vectorstore_index}/{total_vectorstores}] "
                            f"ERROR while building vectorstore after {format_duration(vectorstore_seconds)}: {e}",
                            flush=True,
                        )

                        # Log failed calls so the result file still has one row per planned experiment.
                        for pending in pending_calls:
                            retrieval_metrics = empty_retrieval_metrics(pending["question_index"])

                            row = {
                                "status": "vectorstore_error",
                                "error": str(e),
                                "chunk_method": chunk_method,
                                "chunk_size": chunk_size,
                                "chunk_overlap": chunk_overlap,
                                "num_chunks": None,
                                "prompt_name": pending["prompt_name"],
                                "k": pending["k"],
                                "question": pending["question"],
                                "question_index": pending["question_index"],
                                "answer": f"ERROR: {e}",
                                "answer_length": 0,
                                **retrieval_metrics,
                                "vectorstore_index": vectorstore_index,
                                "total_vectorstores": total_vectorstores,
                                "completed_calls": len(completed_keys) + 1,
                                "total_calls": total_calls,
                                "vectorstore_seconds": round(vectorstore_seconds, 2),
                                "llm_seconds": None,
                                "elapsed_seconds": round(time.time() - start_time, 2),
                                "estimated_remaining_seconds": None,
                            }

                            rows.append(row)
                            completed_keys.add(pending["key"])
                            completed_this_run += 1

                        if partial_results_path is not None:
                            pd.DataFrame(rows).to_csv(partial_results_path, index=False)
                            print(f"Saved progress to {partial_results_path}", flush=True)

                        continue

                    for pending in pending_calls:
                        current_call_number = len(completed_keys) + 1
                        elapsed = time.time() - start_time

                        if completed_this_run > 0:
                            average_seconds = elapsed / completed_this_run
                            estimated_remaining = average_seconds * (total_calls - len(completed_keys))
                            eta_text = format_duration(estimated_remaining)
                        else:
                            estimated_remaining = None
                            eta_text = "?"

                        print(
                            f"[Call {current_call_number}/{total_calls}] "
                            f"method={chunk_method}, size={chunk_size}, overlap={chunk_overlap}, "
                            f"prompt={pending['prompt_name']}, k={pending['k']}, "
                            f"question={pending['question_index']}/{len(questions)} | "
                            f"elapsed={format_duration(elapsed)}, ETA={eta_text}",
                            flush=True,
                        )

                        call_started = time.time()
                        retrieved_docs = []

                        try:
                            # Generate answer and keep retrieved documents for evaluation.
                            answer, retrieved_docs = answer_question(
                                question=pending["question"],
                                vectorstore=vectorstore,
                                llm=llm,
                                k=pending["k"],
                                system_prompt=pending["prompt_text"],
                            )

                            llm_seconds = time.time() - call_started
                            retrieval_metrics = get_retrieval_metrics(
                                retrieved_docs=retrieved_docs,
                                question_index=pending["question_index"],
                            )

                            row = {
                                "status": "ok",
                                "error": "",
                                "chunk_method": chunk_method,
                                "chunk_size": chunk_size,
                                "chunk_overlap": chunk_overlap,
                                "num_chunks": len(chunked),
                                "prompt_name": pending["prompt_name"],
                                "k": pending["k"],
                                "question": pending["question"],
                                "question_index": pending["question_index"],
                                "answer": answer,
                                "answer_length": len(answer),
                                **retrieval_metrics,
                                "vectorstore_index": vectorstore_index,
                                "total_vectorstores": total_vectorstores,
                                "completed_calls": current_call_number,
                                "total_calls": total_calls,
                                "vectorstore_seconds": round(vectorstore_seconds, 2),
                                "llm_seconds": round(llm_seconds, 2),
                                "elapsed_seconds": round(time.time() - start_time, 2),
                                "estimated_remaining_seconds": round(estimated_remaining, 2) if estimated_remaining is not None else None,
                            }

                        except Exception as e:
                            llm_seconds = time.time() - call_started

                            # If retrieval succeeded before the error, preserve those metrics.
                            if retrieved_docs:
                                retrieval_metrics = get_retrieval_metrics(
                                    retrieved_docs=retrieved_docs,
                                    question_index=pending["question_index"],
                                )
                            else:
                                retrieval_metrics = empty_retrieval_metrics(pending["question_index"])

                            row = {
                                "status": "llm_error",
                                "error": str(e),
                                "chunk_method": chunk_method,
                                "chunk_size": chunk_size,
                                "chunk_overlap": chunk_overlap,
                                "num_chunks": len(chunked),
                                "prompt_name": pending["prompt_name"],
                                "k": pending["k"],
                                "question": pending["question"],
                                "question_index": pending["question_index"],
                                "answer": f"ERROR: {e}",
                                "answer_length": 0,
                                **retrieval_metrics,
                                "vectorstore_index": vectorstore_index,
                                "total_vectorstores": total_vectorstores,
                                "completed_calls": current_call_number,
                                "total_calls": total_calls,
                                "vectorstore_seconds": round(vectorstore_seconds, 2),
                                "llm_seconds": round(llm_seconds, 2),
                                "elapsed_seconds": round(time.time() - start_time, 2),
                                "estimated_remaining_seconds": round(estimated_remaining, 2) if estimated_remaining is not None else None,
                            }

                        rows.append(row)
                        completed_keys.add(pending["key"])
                        completed_this_run += 1

                        print(
                            f"[Call {current_call_number}/{total_calls}] "
                            f"Finished with status={row['status']} in {format_duration(llm_seconds)}.",
                            flush=True,
                        )

                        # Incremental saving protects long experiment runs from data loss.
                        if partial_results_path is not None and (
                            save_every <= 1 or completed_this_run % save_every == 0
                        ):
                            pd.DataFrame(rows).to_csv(partial_results_path, index=False)
                            print(f"Saved progress to {partial_results_path}", flush=True)

    finally:
        # Always save what has been completed, even if the notebook is interrupted.
        if partial_results_path is not None and rows:
            pd.DataFrame(rows).to_csv(partial_results_path, index=False)
            print(f"Final/interrupt-safe progress saved to: {partial_results_path}", flush=True)

    final_df = pd.DataFrame(rows)
    elapsed_total = time.time() - start_time

    print("=" * 80, flush=True)
    print("Experiment grid finished", flush=True)
    print(f"Rows in result dataframe: {len(final_df)}", flush=True)
    print(f"Completed calls total:    {len(completed_keys)}/{total_calls}", flush=True)
    print(f"Completed this run:       {completed_this_run}", flush=True)
    print(f"Elapsed this run:         {format_duration(elapsed_total)}", flush=True)
    if partial_results_path is not None:
        print(f"Latest partial CSV:       {partial_results_path}", flush=True)
    print("=" * 80, flush=True)

    return final_df


print("Done! Progress logging and incremental CSV saving are enabled.")

Done! Progress logging and incremental CSV saving are enabled.


### Experiment setup

- `k = [1, 2, 3, 5, 7, 10]`
- `chunk_method = ["recursive", "markdown"]`
- `chunk_size = [300, 500, 700, 900, 1200]`
- `chunk_overlap = [0, 50, 120, 200]`


In [40]:
# Partial results are saved during the run so the experiment can be resumed later.
partial_results_path = CONFIG["data_dir"] / "rag_experiment_partial_results.csv"

# Run the full grid search over chunking, retrieval depth, and prompt settings.
results_df = run_experiment_grid(
    docs=base_docs,
    embedding_model_name=CONFIG["embedding_model"],
    chroma_root=CONFIG["data_dir"] / "experiment_dbs",
    llm=llm,
    questions=EXPERIMENT_QUESTIONS,
    k_values=[1, 2, 3, 5, 7, 10],
    chunk_methods=["recursive", "markdown"],
    chunk_sizes=[300, 500, 700, 900, 1200],
    chunk_overlaps=[0, 50, 120, 200],
    prompt_variants=PROMPT_VARIANTS,
    partial_results_path=partial_results_path,
    save_every=100,
    resume=True,
)

# Inspect the first rows to verify that answers and metrics were saved correctly.
results_df.head(10)

Loaded 3600 existing rows from uia_ikt_rag_data\rag_experiment_partial_results.csv. Skipping 3600 completed experiment calls.
Starting RAG experiment grid
Embedding device:           cuda
Vectorstore builds planned: 40
LLM calls planned:          3600
Already completed:          3600
Partial CSV:                uia_ikt_rag_data\rag_experiment_partial_results.csv
Save frequency:             every 100 completed call(s)
[Vectorstore 1/40] Skipping uia_ikt_recursive_300_0; all calls already completed.
[Vectorstore 2/40] Skipping uia_ikt_recursive_300_50; all calls already completed.
[Vectorstore 3/40] Skipping uia_ikt_recursive_300_120; all calls already completed.
[Vectorstore 4/40] Skipping uia_ikt_recursive_300_200; all calls already completed.
[Vectorstore 5/40] Skipping uia_ikt_recursive_500_0; all calls already completed.
[Vectorstore 6/40] Skipping uia_ikt_recursive_500_50; all calls already completed.
[Vectorstore 7/40] Skipping uia_ikt_recursive_500_120; all calls already complete

,status,error,chunk_method,chunk_size,chunk_overlap,num_chunks,prompt_name,k,question,question_index,...,is_unsupported_question,comparison_coverage_good,vectorstore_index,total_vectorstores,completed_calls,total_calls,vectorstore_seconds,llm_seconds,elapsed_seconds,estimated_remaining_seconds
0,ok,NaN,recursive,300,0,639,strict_grounded,1,What course should I take if I want to learn AI?,1,...,False,NaN,1,40,1,3600,2.23,0.31,2.55,NaN
1,ok,NaN,recursive,300,0,639,strict_grounded,1,Which course is about software architecture?,2,...,False,NaN,1,40,2,3600,2.23,0.31,2.86,9190.41
2,ok,NaN,recursive,300,0,639,strict_grounded,1,What are the differences between the IKT cours...,3,...,False,False,1,40,3,3600,2.23,0.59,3.46,5158.10
3,ok,NaN,recursive,300,0,639,strict_grounded,1,Which IKT course gives the best job opportunit...,4,...,True,NaN,1,40,4,3600,2.23,0.35,3.82,4149.22
4,ok,NaN,recursive,300,0,639,strict_grounded,1,Which course is most related to cybersecurity ...,5,...,False,NaN,1,40,5,3600,2.23,0.35,4.17,3432.64
5,ok,NaN,recursive,300,0,639,strict_grounded,2,What course should I take if I want to learn AI?,1,...,False,NaN,1,40,6,3600,2.23,0.43,4.60,3002.82
6,ok,NaN,recursive,300,0,639,strict_grounded,2,Which course is about software architecture?,2,...,False,NaN,1,40,7,3600,2.23,0.37,4.98,2759.11
7,ok,NaN,recursive,300,0,639,strict_grounded,2,What are the differences between the IKT cours...,3,...,False,False,1,40,8,3600,2.23,0.87,5.86,2556.14
8,ok,NaN,recursive,300,0,639,strict_grounded,2,Which IKT course gives the best job opportunit...,4,...,True,NaN,1,40,9,3600,2.23,0.35,6.21,2630.85
9,ok,NaN,recursive,300,0,639,strict_grounded,2,Which course is most related to cybersecurity ...,5,...,False,NaN,1,40,10,3600,2.23,0.47,6.68,2478.30


In [41]:

# Save final experiment results for analysis.
results_path = CONFIG["data_dir"] / "rag_experiment_results.csv"
results_df.to_csv(results_path, index=False)

print(f"Saved final experiment results to: {results_path}")
print(f"Partial/progress results are available at: {partial_results_path}")
print(f"Rows saved: {len(results_df)}")


Saved final experiment results to: uia_ikt_rag_data\rag_experiment_results.csv
Partial/progress results are available at: uia_ikt_rag_data\rag_experiment_partial_results.csv
Rows saved: 3600


## Fixed RAG setting for qualitative inspection

In [43]:
# Selected configuration used for the final example outputs in the report.
SELECTED_SETTINGS = {
    "chunk_method": "recursive",
    "chunk_size": 700,
    "chunk_overlap": 120,
    "k": 7,
    "prompt_name": "strict_grounded",
}

# Same five questions used in the main experiment.
SELECTED_QUESTIONS = [
    "What course should I take if I want to learn AI?",
    "Which course is about software architecture?",
    "What are the differences between the IKT courses related to AI and machine learning?",
    "Which IKT course gives the best job opportunities?",
    "Which course is most related to cybersecurity research methods?",
]

print("Building vectorstore with selected settings:")
for key, value in SELECTED_SETTINGS.items():
    print(f"  {key}: {value}")

# Rebuild a vector store using the selected settings for manual inspection.
selected_vectorstore, selected_chunks = rebuild_vectorstore_for_experiment(
    docs=base_docs,
    embedding_model_name=CONFIG["embedding_model"],
    persist_dir=CONFIG["data_dir"] / "selected_setting_db",
    collection_name="selected_recursive_700_overlap_120",
    chunk_method=SELECTED_SETTINGS["chunk_method"],
    chunk_size=SELECTED_SETTINGS["chunk_size"],
    chunk_overlap=SELECTED_SETTINGS["chunk_overlap"],
)

print(f"\nNumber of chunks created: {len(selected_chunks)}")

strict_prompt = PROMPT_VARIANTS.get("strict_grounded", DEFAULT_SYSTEM_PROMPT)

inspection_rows = []

for question_id, question in enumerate(SELECTED_QUESTIONS, start=1):
    print("\n" + "=" * 100)
    print(f"Q{question_id}: {question}")
    print("=" * 100)

    # Generate answer and keep retrieved chunks so the output can be inspected.
    answer, retrieved_docs = answer_question(
        question=question,
        vectorstore=selected_vectorstore,
        llm=llm,
        k=SELECTED_SETTINGS["k"],
        system_prompt=strict_prompt,
    )

    print("\nRetrieved sources:")
    for rank, doc in enumerate(retrieved_docs, start=1):
        meta = doc.metadata
        course_code = meta.get("course_code", "Unknown")
        course_name = meta.get("course_name", "")
        source = meta.get("source", meta.get("url", ""))

        # Print a short preview to check what information the model received.
        preview = doc.page_content.replace("\n", "")
        preview = preview[:220] + "..." if len(preview) > 220 else preview

        print(f"{rank}. {course_code} | {course_name}")
        print(f"   Source: {source}")
        print(f"   Preview: {preview}")

    print("\nAnswer:")
    print(answer)

    # Save the selected example outputs used for report inspection.
    inspection_rows.append({
        "question_id": f"Q{question_id}",
        "question": question,
        "chunk_method": SELECTED_SETTINGS["chunk_method"],
        "chunk_size": SELECTED_SETTINGS["chunk_size"],
        "chunk_overlap": SELECTED_SETTINGS["chunk_overlap"],
        "k": SELECTED_SETTINGS["k"],
        "prompt_name": SELECTED_SETTINGS["prompt_name"],
        "retrieved_course_codes": ", ".join(
            [doc.metadata.get("course_code", "Unknown") for doc in retrieved_docs]
        ),
        "retrieved_course_names": " | ".join(
            [doc.metadata.get("course_name", "") for doc in retrieved_docs]
        ),
        "answer": answer,
    })

inspection_df = pd.DataFrame(inspection_rows)

output_path = CONFIG["data_dir"] / "selected_setting_outputs_recursive_700_overlap_120_k7_strict.csv"
inspection_df.to_csv(output_path, index=False)

print("\n" + "=" * 100)
print(f"Saved selected-setting outputs to: {output_path}")
print("=" * 100)

display(inspection_df[[
    "question_id",
    "question",
    "retrieved_course_codes",
    "answer",
]])

Building vectorstore with selected settings:
  chunk_method: recursive
  chunk_size: 700
  chunk_overlap: 120
  k: 7
  prompt_name: strict_grounded
Loading embedding model 'sentence-transformers/all-MiniLM-L6-v2' on device='cuda' with batch_size=256

Number of chunks created: 276

Q1: What course should I take if I want to learn AI?

Retrieved sources:
1. IKT725 | Deep Reinforcement Learning
   Source: https://www.uia.no/studier/emner/2026/var/ikt725.html
   Preview: Upon successful completion of this course, the students are expected to: have a) knowledge of the main features of Deep Reinforcement Learning (DRL) that distinguishes it from other standard machine learning methods, and...
2. IKT112 | Concepts of Machine Learning
   Source: https://www.uia.no/studier/emner/2026/var/ikt112.html
   Preview: IKT112 Concepts of Machine Learning (Vår 2026) - Universitetet i AgderEmnet er tilknyttet følgende studieprogramKunstig intelligens, 5-årig masterprogramUndervisningsspråkEnglishLæringsu

,question_id,question,retrieved_course_codes,answer
0,Q1,What course should I take if I want to learn AI?,"IKT725, IKT112, IKT725, IKT725, IKT724, IKT112...","Based on the context provided, it seems that y..."
1,Q2,Which course is about software architecture?,"IKT723, IKT626, IKT103, IKT101, IKT300, IKT218...","Based on the provided context, the course rela..."
2,Q3,What are the differences between the IKT cours...,"IKT112, IKT724, IKT727, IKT112, IKT710, IKT112...","Based on the provided context, here's a compar..."
3,Q4,Which IKT course gives the best job opportunit...,"IKT626, IKT725, IKT724, IKT720, IKT723, IKT518...","Unfortunately, the context does not provide an..."
4,Q5,Which course is most related to cybersecurity ...,"IKT626, IKT448, IKT523, IKT525, IKT519, IKT447...","Based on the context, the course that is most ..."


## Create tables for analysis and report

In [45]:
import pandas as pd
from pathlib import Path

RESULTS_PATH = Path("uia_ikt_rag_data/rag_experiment_results.csv")
df = pd.read_csv(RESULTS_PATH)

# All rows are OK
ok = df.copy()

bool_cols = [
    "contains_primary_expected",
    "contains_any_expected",
    "is_unsupported_question",
    "comparison_coverage_good",
]

for col in bool_cols:
    if col in ok.columns:
        ok[col] = (
            ok[col]
            .astype(str)
            .str.strip()
            .str.lower()
            .map({
                "true": True,
                "false": False,
                "1": True,
                "0": False,
                "nan": False,
                "none": False,
                "": False,
            })
            .fillna(False)
            .astype(bool)
        )

supported = ok[ok["is_unsupported_question"] == False].copy()

def pct(x):
    return round(x * 100, 1)


# Table 1: Effect of retrieval depth k
table_k = (
    supported
    .groupby("k")
    .agg(
        contains_primary_expected_pct=("contains_primary_expected", lambda x: pct(x.mean())),
        contains_any_expected_pct=("contains_any_expected", lambda x: pct(x.mean())),
        avg_num_sources=("num_sources", "mean"),
        avg_duplicate_ratio=("duplicate_ratio", "mean"),
        avg_answer_length=("answer_length", "mean"),
        runs=("status", "count"),
    )
    .reset_index()
)

table_k["avg_num_sources"] = table_k["avg_num_sources"].round(2)
table_k["avg_duplicate_ratio"] = table_k["avg_duplicate_ratio"].round(3)
table_k["avg_answer_length"] = table_k["avg_answer_length"].round(0).astype(int)

display(table_k)


# Table 2: Effect of chunking method
table_chunk_method = (
    supported
    .groupby("chunk_method")
    .agg(
        contains_primary_expected_pct=("contains_primary_expected", lambda x: pct(x.mean())),
        contains_any_expected_pct=("contains_any_expected", lambda x: pct(x.mean())),
        avg_num_sources=("num_sources", "mean"),
        avg_duplicate_ratio=("duplicate_ratio", "mean"),
        avg_answer_length=("answer_length", "mean"),
        runs=("status", "count"),
    )
    .reset_index()
)

table_chunk_method["avg_num_sources"] = table_chunk_method["avg_num_sources"].round(2)
table_chunk_method["avg_duplicate_ratio"] = table_chunk_method["avg_duplicate_ratio"].round(3)
table_chunk_method["avg_answer_length"] = table_chunk_method["avg_answer_length"].round(0).astype(int)

display(table_chunk_method)


# Table 3: Effect of chunk size
table_chunk_size = (
    supported
    .groupby("chunk_size")
    .agg(
        contains_primary_expected_pct=("contains_primary_expected", lambda x: pct(x.mean())),
        contains_any_expected_pct=("contains_any_expected", lambda x: pct(x.mean())),
        avg_num_sources=("num_sources", "mean"),
        avg_duplicate_ratio=("duplicate_ratio", "mean"),
        runs=("status", "count"),
    )
    .reset_index()
)

table_chunk_size["avg_num_sources"] = table_chunk_size["avg_num_sources"].round(2)
table_chunk_size["avg_duplicate_ratio"] = table_chunk_size["avg_duplicate_ratio"].round(3)

display(table_chunk_size)


# Table 4: Effect of chunk overlap
table_chunk_overlap = (
    supported
    .groupby("chunk_overlap")
    .agg(
        contains_primary_expected_pct=("contains_primary_expected", lambda x: pct(x.mean())),
        contains_any_expected_pct=("contains_any_expected", lambda x: pct(x.mean())),
        avg_num_sources=("num_sources", "mean"),
        avg_duplicate_ratio=("duplicate_ratio", "mean"),
        runs=("status", "count"),
    )
    .reset_index()
)

table_chunk_overlap["avg_num_sources"] = table_chunk_overlap["avg_num_sources"].round(2)
table_chunk_overlap["avg_duplicate_ratio"] = table_chunk_overlap["avg_duplicate_ratio"].round(3)

display(table_chunk_overlap)


# Table 5: Effect of prompt variant
table_prompt = (
    supported
    .groupby("prompt_name")
    .agg(
        contains_primary_expected_pct=("contains_primary_expected", lambda x: pct(x.mean())),
        contains_any_expected_pct=("contains_any_expected", lambda x: pct(x.mean())),
        avg_answer_length=("answer_length", "mean"),
        avg_llm_seconds=("llm_seconds", "mean"),
        runs=("status", "count"),
    )
    .reset_index()
)

table_prompt["avg_answer_length"] = table_prompt["avg_answer_length"].round(0).astype(int)
table_prompt["avg_llm_seconds"] = table_prompt["avg_llm_seconds"].round(2)

display(table_prompt)


# Table 6: Question-specific summary
table_question = (
    ok
    .groupby(["question_index", "question"])
    .agg(
        unsupported_question=("is_unsupported_question", "first"),
        contains_primary_expected_pct=("contains_primary_expected", lambda x: pct(x.mean())),
        contains_any_expected_pct=("contains_any_expected", lambda x: pct(x.mean())),
        avg_num_sources=("num_sources", "mean"),
        avg_duplicate_ratio=("duplicate_ratio", "mean"),
        avg_answer_length=("answer_length", "mean"),
        runs=("status", "count"),
    )
    .reset_index()
)

table_question["avg_num_sources"] = table_question["avg_num_sources"].round(2)
table_question["avg_duplicate_ratio"] = table_question["avg_duplicate_ratio"].round(3)
table_question["avg_answer_length"] = table_question["avg_answer_length"].round(0).astype(int)

table_question.loc[
    table_question["unsupported_question"] == True,
    ["contains_primary_expected_pct", "contains_any_expected_pct"]
] = None

display(table_question)


# Table 7: Best configurations
table_best_configs = (
    supported
    .groupby(["chunk_method", "chunk_size", "chunk_overlap", "k", "prompt_name"])
    .agg(
        contains_primary_expected_pct=("contains_primary_expected", lambda x: pct(x.mean())),
        contains_any_expected_pct=("contains_any_expected", lambda x: pct(x.mean())),
        avg_num_sources=("num_sources", "mean"),
        avg_duplicate_ratio=("duplicate_ratio", "mean"),
        avg_answer_length=("answer_length", "mean"),
        runs=("status", "count"),
    )
    .reset_index()
    .sort_values(
        by=[
            "contains_primary_expected_pct",
            "contains_any_expected_pct",
            "avg_duplicate_ratio",
        ],
        ascending=[False, False, True],
    )
    .head(10)
)

table_best_configs["avg_num_sources"] = table_best_configs["avg_num_sources"].round(2)
table_best_configs["avg_duplicate_ratio"] = table_best_configs["avg_duplicate_ratio"].round(3)
table_best_configs["avg_answer_length"] = table_best_configs["avg_answer_length"].round(0).astype(int)

display(table_best_configs)


# Save tables
output_dir = Path("analysis_tables")
output_dir.mkdir(exist_ok=True)

table_k.to_csv(output_dir / "table_k.csv", index=False)
table_chunk_method.to_csv(output_dir / "table_chunk_method.csv", index=False)
table_chunk_size.to_csv(output_dir / "table_chunk_size.csv", index=False)
table_chunk_overlap.to_csv(output_dir / "table_chunk_overlap.csv", index=False)
table_prompt.to_csv(output_dir / "table_prompt.csv", index=False)
table_question.to_csv(output_dir / "table_question.csv", index=False)
table_best_configs.to_csv(output_dir / "table_best_configs.csv", index=False)

,k,contains_primary_expected_pct,contains_any_expected_pct,avg_num_sources,avg_duplicate_ratio,avg_answer_length,runs
0,1,44.4,67.5,1.00,0.000,496,480
1,2,59.4,75.0,1.76,0.122,644,480
2,3,65.0,75.0,2.29,0.235,689,480
3,5,75.6,80.6,3.71,0.258,822,480
4,7,77.5,82.5,5.18,0.260,1224,480
5,10,84.4,85.0,7.28,0.272,877,480


,chunk_method,contains_primary_expected_pct,contains_any_expected_pct,avg_num_sources,avg_duplicate_ratio,avg_answer_length,runs
0,markdown,66.5,73.1,3.50,0.215,737,1440
1,recursive,69.0,82.1,3.58,0.167,847,1440


,chunk_size,contains_primary_expected_pct,contains_any_expected_pct,avg_num_sources,avg_duplicate_ratio,runs
0,300,58.9,71.4,3.28,0.234,576
1,500,70.8,76.6,3.42,0.208,576
2,700,71.9,80.7,3.66,0.165,576
3,900,72.9,80.2,3.61,0.182,576
4,1200,64.1,79.2,3.72,0.166,576


,chunk_overlap,contains_primary_expected_pct,contains_any_expected_pct,avg_num_sources,avg_duplicate_ratio,runs
0,0,67.5,77.1,3.55,0.191,720
1,50,68.3,77.5,3.55,0.189,720
2,120,66.7,77.9,3.56,0.185,720
3,200,68.3,77.9,3.49,0.199,720


,prompt_name,contains_primary_expected_pct,contains_any_expected_pct,avg_answer_length,avg_llm_seconds,runs
0,compare_courses,67.7,77.6,1014,1.11,960
1,friendly_advisor,67.7,77.6,819,0.91,960
2,strict_grounded,67.7,77.6,542,0.66,960


,question_index,question,unsupported_question,contains_primary_expected_pct,contains_any_expected_pct,avg_num_sources,avg_duplicate_ratio,avg_answer_length,runs
0,1,What course should I take if I want to learn AI?,False,94.2,97.5,2.95,0.262,774,720
1,2,Which course is about software architecture?,False,15.4,15.4,3.76,0.170,725,720
2,3,What are the differences between the IKT cours...,False,100.0,100.0,3.20,0.287,1131,720
3,4,Which IKT course gives the best job opportunit...,True,NaN,NaN,4.53,0.017,522,720
4,5,Which course is most related to cybersecurity ...,False,61.3,97.5,4.25,0.046,537,720


,chunk_method,chunk_size,chunk_overlap,k,prompt_name,contains_primary_expected_pct,contains_any_expected_pct,avg_num_sources,avg_duplicate_ratio,avg_answer_length,runs
549,recursive,700,120,5,compare_courses,100.0,100.0,4.25,0.150,782,4
550,recursive,700,120,5,friendly_advisor,100.0,100.0,4.25,0.150,823,4
551,recursive,700,120,5,strict_grounded,100.0,100.0,4.25,0.150,654,4
567,recursive,700,200,5,compare_courses,100.0,100.0,4.25,0.150,779,4
568,recursive,700,200,5,friendly_advisor,100.0,100.0,4.25,0.150,794,4
569,recursive,700,200,5,strict_grounded,100.0,100.0,4.25,0.150,656,4
663,recursive,1200,0,10,compare_courses,100.0,100.0,8.25,0.175,1022,4
664,recursive,1200,0,10,friendly_advisor,100.0,100.0,8.25,0.175,1056,4
665,recursive,1200,0,10,strict_grounded,100.0,100.0,8.25,0.175,620,4
681,recursive,1200,50,10,compare_courses,100.0,100.0,8.25,0.175,1022,4
